# BREACHZERO - Training and Evaluation

Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.
Uploads trained models to Hugging Face Hub (optional).

In [ ]:
![ -d /content/BREACHZERO ] || git clone https://github.com/Priyanshu-2308/BREACHZERO.git /content/BREACHZERO

In [ ]:
%cd /content/BREACHZERO
!git pull --rebase origin main

In [ ]:
# Colab's own torch/transformers are used as-is. Only sklearn gets a floor, because
# models/baseline.joblib was pickled with scikit-learn 1.9.1.
!pip install -q "scikit-learn>=1.9"

In [ ]:
import joblib, sklearn, torch, transformers
print("sklearn", sklearn.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set the runtime to T4")

# fail fast: make sure the baseline loads before spending 15 minutes on training
pipe = joblib.load("models/baseline.joblib")
print("baseline check:", pipe.predict_proba(["ignore previous instructions"])[0][1].round(3))

## Train both models

In [ ]:
!python train.py --out models/distilbert --epochs 3

In [ ]:
from config import MULTI_MODEL
!python train.py --model {MULTI_MODEL} --out models/distilbert_multilingual --epochs 3

## Evaluate

In [ ]:
!python run_eval.py

In [ ]:
import pandas as pd
pd.read_csv("results/main_results.csv")

## Push models to the Hub, fill in MODELS.md

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import pathlib, traceback
from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()["name"]
print("logged in as", user)

repos = {
    "models/distilbert": "breachzero-distilbert",
    "models/distilbert_multilingual": "breachzero-distilbert-multilingual",
}
for folder, name in repos.items():
    if not pathlib.Path(folder, "config.json").exists():
        print("SKIP", folder, "- not trained")
        continue
    repo_id = user + "/" + name
    try:
        api.create_repo(repo_id, repo_type="model", exist_ok=True)
        api.upload_folder(folder_path=folder, repo_id=repo_id, repo_type="model")
        print("pushed", repo_id)
    except Exception:
        traceback.print_exc()

In [ ]:
!python train.py --out /tmp/smoketest --epochs 1 --max_len 64 --limit 32

In [ ]:
%cd /content/BREACHZERO
!git pull --rebase origin main
!grep -c build_args train.py
!python train.py --out /tmp/smoketest --epochs 1 --max_len 64 --limit 32